<a href="https://colab.research.google.com/github/CaioGouveia924/APS-1-/blob/main/APS1_Inadimplencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# APS 1 Inadimplencia

**Disciplina:** Inteligência Artificial Aplicada    
**Integrantes:** Caio André, Maria Vitória, Marina Félix e Pamela Machado.

Nesta APS será construído um **sistema baseado em conhecimento com inferência probabilística**. A atividade retoma o percurso da Aula 07:

1. representar hipóteses e evidências;
2. calcular uma atualização probabilística manualmente;
3. comparar hipóteses concorrentes;
4. implementar um motor de inferência;
5. combinar várias evidências;
6. discutir os limites do modelo.

O objetivo não é criar um sistema financeiro real. Todos os dados e parâmetros desta atividade são fictícios.

# 1 · Problema e hipóteses

O sistema analisa uma proposta fictícia e considera três resultados possíveis para os **12 meses seguintes**.

| Hipótese | Significado |
|---|---|
| `regular` | nenhuma parcela ultrapassa 30 dias de atraso |
| `atraso_moderado` | ocorre atraso entre 31 e 60 dias, mas nenhum atraso superior a 60 dias |
| `inadimplencia` | ocorre pelo menos uma parcela com mais de 60 dias de atraso |

Para esta atividade, as três hipóteses são tratadas como **mutuamente exclusivas** e representam todas as possibilidades consideradas pelo modelo.

As probabilidades iniciais são:

| Hipótese | Prior |
|---|---:|
| `regular` | 0,55 |
| `atraso_moderado` | 0,30 |
| `inadimplencia` | 0,15 |

### Questão 1

Explique, em uma ou duas frases, o que significa dizer que o prior de `inadimplencia` é 0,15.

**Resposta:** Significa que, antes de observar qualquer evidência sobre a proposta, o modelo atribui 15% de probabilidade de que ela termine em inadimplência (parcela com mais de 60 dias de atraso) nos 12 meses seguintes. É a crença inicial, que será atualizada conforme as evidências forem observadas.

# 2 · Evidências do domínio

O sistema utiliza seis evidências possíveis.

| ID | Evidência observada quando... |
|---|---|
| **E1** | houve atraso anterior superior a 30 dias |
| **E2** | parcelas existentes + nova parcela ultrapassam 40% da renda |
| **E3** | a reserva cobre pelo menos 3 meses de despesas essenciais |
| **E4** | houve recebimento de renda em todos os últimos 12 meses |
| **E5** | cheque especial foi usado em 10 ou mais dias dos últimos 30 |
| **E6** | o prazo solicitado é superior a 24 meses |

A base de conhecimento associa cada evidência às três hipóteses.

In [1]:
base = {
    "regular": {
        "prior": 0.55,
        "evidencias": {
            "E1": 0.1,
            "E2": 0.2,
            "E3": 0.7,
            "E4": 0.85,
            "E5": 0.1,
            "E6": 0.35,
        },
    },
    "atraso_moderado": {
        "prior": 0.3,
        "evidencias": {
            "E1": 0.4,
            "E2": 0.5,
            "E3": 0.4,
            "E4": 0.65,
            "E5": 0.35,
            "E6": 0.5,
        },
    },
    "inadimplencia": {
        "prior": 0.15,
        "evidencias": {
            "E1": 0.7,
            "E2": 0.75,
            "E3": 0.15,
            "E4": 0.45,
            "E5": 0.6,
            "E6": 0.6,
        },
    },
}

## Leitura da base

Por exemplo:

- $P(E1 \mid regular)=0{,}10$
- $P(E1 \mid atraso\_moderado)=0{,}40$
- $P(E1 \mid inadimplencia)=0{,}70$

Esses valores respondem à pergunta:

> Se uma determinada hipótese for verdadeira, com que frequência esperamos observar E1?

Eles **não** são os posteriores procurados.

### Questão 2

Explique a diferença entre:

- $P(E1 \mid inadimplencia)$
- $P(inadimplencia \mid E1)$

**Resposta:** $P(E1 \mid inadimplencia)$ é a verossimilhança: entre os casos que terminam em inadimplência, qual a frequência esperada de observar atraso anterior superior a 30 dias (0,70 na base). Já $P(inadimplencia \mid E1)$ é o posterior: dado que E1 foi observado, qual a probabilidade de a proposta terminar em inadimplência (0,375 na seção 3). Trocar uma pela outra é a falácia da condicional invertida. Para ir de uma à outra é preciso combinar a verossimilhança com o prior e normalizar pelas demais hipóteses (Teorema de Bayes).

# 3 · Uma evidência e três hipóteses

Considere inicialmente apenas a evidência **E1: atraso anterior superior a 30 dias**.

Para cada hipótese, calcule um valor proporcional ao posterior:

$$
score(H)=P(H)\times P(E1\mid H)
$$

Complete a tabela manualmente.

| Hipótese | Prior | $P(E1\mid H)$ | Score |
|---|---:|---:|---:|
| `regular` | 0,55 | 0,10 | 0,055 |
| `atraso_moderado` | 0,30 | 0,40 | 0,120 |
| `inadimplencia` | 0,15 | 0,70 | 0,105 |

Depois:

1. some os três scores;
2. normalize cada score dividindo-o pelo total;
3. registre o posterior das três hipóteses;
4. indique qual hipótese tem o maior posterior;
5. explique por que a hipótese com maior $P(E1\mid H)$ não necessariamente precisa ter o maior posterior.

### Registro

**Soma dos scores:** 0,055 + 0,120 + 0,105 = **0,280**.

| Hipótese | Posterior |
|---|---:|
| `regular` | 0,055 / 0,280 ≈ 0,1964 (19,64%) |
| `atraso_moderado` | 0,120 / 0,280 ≈ 0,4286 (42,86%) |
| `inadimplencia` | 0,105 / 0,280 = 0,3750 (37,50%) |

**Hipótese mais provável:** `atraso_moderado`.

**Interpretação:** `inadimplencia` tem a maior verossimilhança (P(E1 | H) = 0,70), mas parte de um prior baixo (0,15). O posterior depende do produto prior × verossimilhança, e não só da verossimilhança. `atraso_moderado` combina um prior bem maior (0,30) com uma verossimilhança ainda alta (0,40), por isso o score dela (0,120) supera o de `inadimplencia` (0,105). Ou seja, ver E1 aumenta bastante a chance de inadimplência (de 15% para 37,5%), mas não o suficiente para torná-la a hipótese mais provável.

# 4 · Implementar o motor de inferência

Agora transforme o cálculo anterior em uma função genérica.

A função deve:

1. receber a base de conhecimento;
2. receber uma lista de evidências observadas;
3. iniciar o score de cada hipótese pelo prior;
4. multiplicar o score por $P(E\mid H)$ para cada evidência observada;
5. somar os scores;
6. normalizar os valores;
7. retornar os posteriores e a hipótese com maior posterior.

Implemente a função a partir do contrato acima.

In [ ]:
def inferir(base, evidencias_observadas):
    """Retorna (posteriores, predicao)."""
    # 1) score inicial = prior; 2) multiplica pela verossimilhança de cada evidência observada
    scores = {}
    for hipotese, dados in base.items():
        score = dados["prior"]
        for evidencia in evidencias_observadas:
            score *= dados["evidencias"][evidencia]
        scores[hipotese] = score

    # 3) normaliza para que os posteriores somem 1
    total = sum(scores.values())
    posteriores = {h: s / total for h, s in scores.items()}

    # 4) hipótese com maior posterior
    predicao = max(posteriores, key=posteriores.get)
    return posteriores, predicao

## Verificação mínima do motor

O caso com apenas E1 deve reproduzir o cálculo manual da seção anterior.

Execute os testes somente depois de implementar `inferir`.

In [ ]:
posteriores_e1, predicao_e1 = inferir(base, ["E1"])

assert abs(sum(posteriores_e1.values()) - 1.0) < 1e-9
assert abs(posteriores_e1["regular"] - 0.19642857142857145) < 1e-9
assert abs(posteriores_e1["atraso_moderado"] - 0.42857142857142855) < 1e-9
assert abs(posteriores_e1["inadimplencia"] - 0.375) < 1e-9
assert predicao_e1 == "atraso_moderado"

print("Teste do motor aprovado.")

Teste do motor aprovado.


# 5 · Dos dados da proposta para as evidências

Agora o sistema precisa transformar dados de uma proposta nas evidências E1 a E6.

As propostas fictícias já estão fornecidas.

In [ ]:
propostas = [
    {
        "id": "P01",
        "renda_mensal": 5000,
        "parcelas_existentes": 1300,
        "nova_parcela": 900,
        "reserva": 12000,
        "despesas_essenciais": 3000,
        "maior_atraso_12m": 45,
        "meses_com_renda_12m": 12,
        "dias_cheque_30d": 6,
        "prazo_meses": 36,
    },
    {
        "id": "P02",
        "renda_mensal": 6000,
        "parcelas_existentes": 500,
        "nova_parcela": 700,
        "reserva": 18000,
        "despesas_essenciais": 3000,
        "maior_atraso_12m": 0,
        "meses_com_renda_12m": 12,
        "dias_cheque_30d": 0,
        "prazo_meses": 18,
    },
    {
        "id": "P03",
        "renda_mensal": 4500,
        "parcelas_existentes": 900,
        "nova_parcela": 1000,
        "reserva": 6000,
        "despesas_essenciais": 2500,
        "maior_atraso_12m": 35,
        "meses_com_renda_12m": 11,
        "dias_cheque_30d": 12,
        "prazo_meses": 30,
    },
]

## Implemente `extrair_evidencias`

A função deve receber uma proposta e retornar uma lista contendo **somente as evidências observadas**.

Exemplo de raciocínio:

- se `maior_atraso_12m > 30`, incluir `"E1"`;
- se a condição não for satisfeita, `"E1"` simplesmente não aparece na lista.

Use exatamente as seis regras apresentadas na seção 2.

In [ ]:
def extrair_evidencias(proposta):
    """Transforma os dados da proposta em uma lista de evidências observadas."""
    evidencias = []

    # E1: atraso anterior superior a 30 dias
    if proposta["maior_atraso_12m"] > 30:
        evidencias.append("E1")

    # E2: parcelas existentes + nova parcela ultrapassam 40% da renda
    comprometimento = (proposta["parcelas_existentes"] + proposta["nova_parcela"]) / proposta["renda_mensal"]
    if comprometimento > 0.40:
        evidencias.append("E2")

    # E3: reserva cobre pelo menos 3 meses de despesas essenciais
    meses_reserva = proposta["reserva"] / proposta["despesas_essenciais"]
    if meses_reserva >= 3:
        evidencias.append("E3")

    # E4: renda recebida em todos os últimos 12 meses
    if proposta["meses_com_renda_12m"] == 12:
        evidencias.append("E4")

    # E5: cheque especial usado em 10 ou mais dias dos últimos 30
    if proposta["dias_cheque_30d"] >= 10:
        evidencias.append("E5")

    # E6: prazo solicitado superior a 24 meses
    if proposta["prazo_meses"] > 24:
        evidencias.append("E6")

    return evidencias

## Conferência manual de P01

Antes de executar a função, determine quais evidências devem ser observadas em P01.

Registre também os dois cálculos necessários:

- comprometimento da renda;
- quantidade de meses cobertos pela reserva.

**Comprometimento:** (1300 + 900) / 5000 = 2200 / 5000 = 0,44 = **44%** (acima de 40%, logo E2 é observada).  
**Meses de reserva:** 12000 / 3000 = **4 meses** (pelo menos 3, logo E3 é observada).  
**Evidências esperadas:** E1 (atraso de 45 dias > 30), E2, E3, E4 (12 meses com renda), E6 (prazo de 36 meses > 24). E5 não ocorre (apenas 6 dias de cheque especial). Resultado: `["E1", "E2", "E3", "E4", "E6"]`.

In [ ]:
evidencias_p01 = extrair_evidencias(propostas[0])

assert evidencias_p01 == ["E1", "E2", "E3", "E4", "E6"]

print("Evidências de P01:", evidencias_p01)

Evidências de P01: ['E1', 'E2', 'E3', 'E4', 'E6']


# 6 · Inferência com várias evidências

Agora analise as três propostas usando todas as evidências observadas.

Para cada proposta, registre:

- evidências observadas;
- posteriores;
- hipótese com maior posterior.

In [ ]:
resultados = []

for proposta in propostas:
    evidencias = extrair_evidencias(proposta)
    posteriores, predicao = inferir(base, evidencias)

    resultados.append({
        "id": proposta["id"],
        "evidencias": evidencias,
        "posteriores": posteriores,
        "predicao": predicao,
    })

for resultado in resultados:
    print("\n", resultado["id"])
    print("Evidências:", resultado["evidencias"])
    for hipotese, probabilidade in resultado["posteriores"].items():
        print(f"{hipotese}: {probabilidade:.2%}")
    print("Predição:", resultado["predicao"])


 P01
Evidências: ['E1', 'E2', 'E3', 'E4', 'E6']
regular: 17.25%
atraso_moderado: 58.73%
inadimplencia: 24.02%
Predição: atraso_moderado

 P02
Evidências: ['E3', 'E4']
regular: 78.78%
atraso_moderado: 18.78%
inadimplencia: 2.44%
Predição: regular

 P03
Evidências: ['E1', 'E2', 'E5', 'E6']
regular: 0.98%
atraso_moderado: 26.76%
inadimplencia: 72.26%
Predição: inadimplencia


**Resumo dos resultados:**

| Proposta | Evidências | regular | atraso_moderado | inadimplencia | Predição |
|---|---|---:|---:|---:|---|
| P01 | E1, E2, E3, E4, E6 | 17,25% | 58,73% | 24,02% | `atraso_moderado` |
| P02 | E3, E4 | 78,78% | 18,78% | 2,44% | `regular` |
| P03 | E1, E2, E5, E6 | 0,98% | 26,76% | 72,26% | `inadimplencia` |

P02 reúne evidências favoráveis (reserva alta e renda estável) e nenhum sinal de risco, então `regular` domina. P03 acumula quatro sinais de risco (atraso anterior, alto comprometimento, cheque especial e prazo longo) e `inadimplencia` passa a ser a mais provável. P01 tem sinais mistos, e o resultado fica concentrado em `atraso_moderado`.

# 7 · Uma nova evidência pode alterar a predição

A Aula 07 mostra que uma segunda evidência pode modificar a distribuição e até alterar a hipótese mais provável.

Vamos observar esse comportamento em P01.

Calcule duas versões:

1. usando apenas `E1`;
2. usando todas as evidências extraídas de P01.

Depois compare os resultados.

In [ ]:
posteriores_so_e1, predicao_so_e1 = inferir(base, ["E1"])

evidencias_p01 = extrair_evidencias(propostas[0])
posteriores_completos, predicao_completa = inferir(base, evidencias_p01)

print("Somente E1:")
print(posteriores_so_e1)
print("Predição:", predicao_so_e1)

print("\nTodas as evidências de P01:")
print(posteriores_completos)
print("Predição:", predicao_completa)

Somente E1:
{'regular': 0.19642857142857142, 'atraso_moderado': 0.4285714285714285, 'inadimplencia': 0.37499999999999994}
Predição: atraso_moderado

Todas as evidências de P01:
{'regular': 0.17249461130825197, 'atraso_moderado': 0.587343869127738, 'inadimplencia': 0.2401615195640101}
Predição: atraso_moderado


### Questão 3

Compare as duas distribuições.

1. Quais probabilidades mudaram?
2. A hipótese mais provável mudou?
3. Identifique uma evidência de P01 que favorece relativamente `regular`.
4. Identifique uma evidência de P01 que favorece relativamente `inadimplencia`.
5. Explique por que o sistema não deve interpretar a maior posterior como certeza.

**Resposta:**

1. Todas mudaram. `regular` foi de 19,64% para 17,25%; `atraso_moderado` subiu de 42,86% para 58,73%; `inadimplencia` caiu de 37,50% para 24,02%.
2. Não. Nas duas versões a hipótese mais provável é `atraso_moderado`. O que mudou foi a distribuição: a vantagem de `atraso_moderado` sobre `inadimplencia` aumentou (de cerca de 5 para cerca de 35 pontos percentuais), então a predição ficou mais separada, mas não trocou de hipótese.
3. E3 (reserva de 4 meses) favorece `regular`: P(E3 | regular) = 0,70, contra 0,15 em `inadimplencia`. E4 (renda em todos os 12 meses) também: 0,85 contra 0,45.
4. E1 favorece `inadimplencia`: P(E1 | inadimplencia) = 0,70, contra 0,10 em `regular`. E2 (comprometimento de 44%) também: 0,75 contra 0,20.
5. O posterior é uma probabilidade, não um veredito: mesmo a hipótese vencedora (58,73%) deixa cerca de 41% de probabilidade para as outras duas. Além disso, o resultado depende de priors e verossimilhanças fictícios e da suposição de independência condicional; se esses parâmetros estiverem errados, a probabilidade calculada também estará.

# 8 · Independência condicional e leitura crítica

Ao multiplicar as probabilidades das evidências, o modelo faz uma simplificação importante: assume que as evidências são **condicionalmente independentes dada a hipótese**.

Considere, por exemplo:

- E2: comprometimento da renda acima de 40%;
- E5: uso frequente do cheque especial.

### Questão 4

Responda:

1. Por que E2 e E5 podem estar relacionadas no mundo real?
2. O que pode acontecer se tratarmos duas evidências fortemente relacionadas como independentes?
3. Por que passar nos testes de código não demonstra que as probabilidades do modelo são adequadas para decisões reais?
4. De onde poderiam vir priors e probabilidades condicionais em um sistema real?

**Resposta:**

1. Uma pessoa com a renda muito comprometida com parcelas tem pouca folga no orçamento e tende a recorrer ao cheque especial para fechar o mês. Há uma causa comum (aperto financeiro), então E2 e E5 não variam de forma independente, mesmo dada a hipótese.
2. O modelo conta praticamente a mesma informação duas vezes (dupla contagem). Isso exagera o peso do conjunto e produz posteriores mais extremos do que deveriam ser, ou seja, excesso de confiança.
3. Os testes só verificam se o código executa corretamente a conta definida (por exemplo, se os posteriores somam 1 e reproduzem o cálculo manual). Eles não verificam se os priors e as verossimilhanças refletem a realidade, se o modelo está calibrado ou se funciona em dados novos.
4. De dados históricos de carteiras de crédito (frequências observadas de cada desfecho e de cada evidência por desfecho), de bases de bureaus de crédito, de estudos estatísticos e, quando faltam dados, de elicitação com especialistas. Em qualquer caso, é preciso validar e recalibrar com dados fora da amostra e atualizar periodicamente.

# 9 · Entrega

Entregar o notebook com:

- respostas das questões conceituais;
- implementação de `inferir`;
- implementação de `extrair_evidencias`;
- execução das três propostas;
- comparação entre uma e várias evidências;
- análise crítica da independência condicional.

## Critérios de avaliação

| Dimensão | O que será observado |
|---|---|
| Fundamento probabilístico | interpretação correta de prior, verossimilhança, score e posterior |
| Modelagem | transformação correta dos dados em E1 a E6 |
| Implementação | motor genérico de inferência e integração com as propostas |
| Análise | comparação coerente entre hipóteses e evidências |
| Leitura crítica | compreensão dos limites da independência condicional e do significado da predição |

## Material de estudo

Aula 07 · Raciocínio sob incerteza:  
https://pfransozi.github.io/ia-aplicada-2026-02/aula-07.html